# 03. Flash Attention, Online Softmax, and Flex Attention

- **Category:** 05_architectures / 04_transformers_and_heads
- **Lineage & Citations:** Maxim Milakov & Natalia Gimelshteyn, 2018 (*Online Softmax*); Tri Dao et al., 2022 (*FlashAttention*); Tri Dao, 2023 (*FlashAttention-2*); Horace Hooper et al., 2024 (*FlexAttention*).
- **Purpose:** Overcomes the GPU memory bandwidth wall by tiling computation in fast SRAM, using incremental Online Softmax to achieve $\mathcal{O}(N)$ IO complexity without materializing intermediate $T \times T$ attention matrices.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Transformer Heads Induction OV QK Circuits MTP and Registers](02_transformer_heads_induction_ov_qk_circuits_mtp_and_registers.ipynb) $\to$ `🔗 Downstream Syntheses:` [Sparse Sliding Window and Sink Attention](04_sparse_sliding_window_and_sink_attention.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> - **Ultra-Short Sequences ($T < 128$):** Kernel launch overhead dominates; standard unfused GEMMs are faster on tiny sequence lengths.
> - **Dense Attention Map Inspection:** Because FlashAttention never materializes the attention matrix $P$, use standard attention if explicitly saving attention weight heatmaps for interpretability.

## Mathematical Foundations & Governing Equations

### 1. Tiled Online Softmax Formulation
Given key-value block $j$, score tile $S^{(j)} = Q (K^{(j)})^\top \cdot \text{scale}$, and running maximum $m$:
$$
m^{(j)} = \max(m^{(j-1)}, \max S^{(j)}), \quad \alpha = \exp(m^{(j-1)} - m^{(j)}), \quad P^{(j)} = \exp(S^{(j)} - m^{(j)})
$$
Running normalizer and output accumulation:
$$
l^{(j)} = \alpha l^{(j-1)} + \sum P^{(j)}, \quad O^{(j)} = \frac{\alpha l^{(j-1)} O^{(j-1)} + P^{(j)} V^{(j)}}{l^{(j)}}
$$

In [ ]:
import math
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/05_architectures/04_transformers_and_heads/03_flash_attention_online_softmax_and_flex_attention.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)

q, k, v = weights["q"], weights["k"], weights["v"]
scale = float(weights["scale"][0])
block_size = int(weights["block_size"][0])
print(f"Loaded {len(weights)} golden tensors on {env.device}, Q: {list(q.shape)}, K: {list(k.shape)}, block: {block_size}")

In [ ]:
def online_softmax_flash_attention(q, k, v, block_size, scale):
    B, T, D = q.shape
    S = k.shape[1]
    out = torch.zeros(B, T, D, device=q.device, dtype=q.dtype)
    m = torch.full((B, T, 1), -float("inf"), device=q.device, dtype=q.dtype)
    l = torch.zeros(B, T, 1, device=q.device, dtype=q.dtype)
    num_blocks = (S + block_size - 1) // block_size
    for j in range(num_blocks):
        start = j * block_size
        end = min(start + block_size, S)
        k_block, v_block = k[:, start:end, :], v[:, start:end, :]
        scores_j = torch.bmm(q, k_block.transpose(1, 2)) * scale
        block_max = scores_j.max(dim=-1, keepdim=True).values
        m_new = torch.maximum(m, block_max)
        alpha = torch.exp(m - m_new)
        p_j = torch.exp(scores_j - m_new)
        l_new = alpha * l + p_j.sum(dim=-1, keepdim=True)
        out = (alpha * l * out + torch.bmm(p_j, v_block)) / l_new
        m, l = m_new, l_new
    return out, m, l

def standard_attention(q, k, v, scale):
    scores = torch.bmm(q, k.transpose(1, 2)) * scale
    return torch.bmm(F.softmax(scores, dim=-1), v)

In [ ]:
# Quantitative Mathematical Assertions & Golden Parity Verification
flash_out, final_m, final_l = online_softmax_flash_attention(q, k, v, block_size, scale)
err_f = float(torch.max(torch.abs(flash_out - weights["flash_out"])))
err_std = float(torch.max(torch.abs(flash_out - weights["std_out"])))
assert err_f < 1e-6, f"Flash output error: {err_f:.2e}"
assert err_std < 1e-6, f"Equivalence to standard attention failed: {err_std:.2e}"

std_ref = standard_attention(q, k, v, scale)
err_ref = float(torch.max(torch.abs(flash_out - std_ref)))
assert err_ref < 1e-6, f"Direct reference verification failed: {err_ref:.2e}"
print(f"Parity Verified: Flash vs Golden={err_f:.2e}, Flash vs Standard={err_std:.2e}")

In [ ]:
import matplotlib.pyplot as plt
from core.benchmark import benchmark_fn, format_benchmark_table

b_flash = benchmark_fn(lambda: online_softmax_flash_attention(q, k, v, block_size, scale), iters=50, name="Tiled FlashAttention")
b_std = benchmark_fn(lambda: standard_attention(q, k, v, scale), iters=50, name="Standard Softmax Attn")
print(format_benchmark_table([b_flash, b_std]))

# Visualization: Output Trajectory & Running Normalizer Convergence
fig, ax = plt.subplots(1, 2, figsize=(8, 2.5), dpi=100)
ax[0].imshow(flash_out[0].detach().numpy(), cmap="viridis", aspect="auto")
ax[0].set_title("FlashAttention Output [T, D]")
ax[1].plot(final_l[0, :, 0].detach().numpy(), marker="o", color="crimson")
ax[1].set_title("Online Normalizer l_t across Queries")
plt.tight_layout()
plt.show()

In [ ]:
# Pathological Misuse: Unscaled Intermediates Leading to Numerical Overflow
# Without subtracting the running maximum m, computing raw exp(scores) blows up to Inf for large logits!
scores_huge = torch.tensor([100.0, 102.0, 105.0])
exp_naive = torch.exp(scores_huge)
# Stable subtraction:
m_safe = scores_huge.max()
exp_safe = torch.exp(scores_huge - m_safe)

print(f"Naive exp without max subtraction: {exp_naive.numpy()} (is_inf: {torch.isinf(exp_naive).any().item()})")
print(f"Safe online softmax exp: {exp_safe.numpy()}")
assert torch.isinf(exp_naive).any(), "Large logits must overflow float32 without online max subtraction!"
print("Demonstrated requirement of online running maximum subtraction for numerical stability.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **IO-Awareness is Decisive:** Shifting bottlenecks from memory bandwidth (HBM) to on-chip compute (SRAM) provides massive real-world speedups.
2. **Online Softmax Enables Constant Memory:** Updating statistics iteratively eliminates the need to allocate intermediate $T \times T$ matrices.
3. **FlexAttention Unifies Flexibility & Speed:** Custom masks and biases compile seamlessly into fused execution kernels.

---
### Cross-Domain Generalisation Matrix
- **Ultra Long-Context LLMs:** Powers 100k+ token document analysis without Out-of-Memory crashes.
- **High-Resolution Diffusion:** Computes self-attention across millions of 2D/3D video tokens.
- **Edge Transformer Acceleration:** Fits full attention computation within strict SRAM budgets on mobile NPUs.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [03_flash_attention_online_softmax_and_flex_attention.md](../../../markdown/05_architectures/04_transformers_and_heads/03_flash_attention_online_softmax_and_flex_attention.md)
- **PyTorch 2.x Module:** [03_flash_attention_online_softmax_and_flex_attention.py](../../../pytorch2/05_architectures/04_transformers_and_heads/03_flash_attention_online_softmax_and_flex_attention.py)
- **Functional JAX Module:** [03_flash_attention_online_softmax_and_flex_attention.py](../../../jax/05_architectures/04_transformers_and_heads/03_flash_attention_online_softmax_and_flex_attention.py)
- **Candle Rust Module:** [03_flash_attention_online_softmax_and_flex_attention.rs](../../../candle/05_architectures/04_transformers_and_heads/03_flash_attention_online_softmax_and_flex_attention.rs)
- **Hardware Kernel:** [03_flash_attention_online_softmax_and_flex_attention.rs](../../../kernels/05_architectures/04_transformers_and_heads/03_flash_attention_online_softmax_and_flex_attention.rs)
- **Universal Golden SafeTensors:** [03_flash_attention_online_softmax_and_flex_attention.safetensors](../../../weights/05_architectures/04_transformers_and_heads/03_flash_attention_online_softmax_and_flex_attention.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous Topic: 02. Transformer Heads Induction OV QK Circuits MTP and Registers](02_transformer_heads_induction_ov_qk_circuits_mtp_and_registers.ipynb) | [Category Index](../../../README.md#05_architectures) | [Next Topic: 04. Sparse Sliding Window and Sink Attention](04_sparse_sliding_window_and_sink_attention.ipynb) &rarr;